<a href="https://colab.research.google.com/github/Mweressa27/hippotherapy-simulator/blob/main/rrur_kinematics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sympy import *
from scipy.optimize import root

# Constants
l_11, l_12, l_13 = symbols("l_{11} l_{12} l_{13}")  # chain n length m
l_21, l_22, l_23 = symbols("l_{21} l_{22} l_{23}")
l_31, l_32, l_33 = symbols("l_{31} l_{32} l_{33}")

D, B, H = symbols("D B H")

thetaA_1, thetaA_2, thetaA_3 = symbols("\\theta_{A1} \\theta_{A2} \\theta_{A3}")  # revolute n chain m
thetaB_1, thetaB_2, thetaB_3 = symbols("\\theta_{B1} \\theta_{B2} \\theta_{B3}")
thetaC_1, thetaC_2, thetaC_3 = symbols("\\theta_{C1} \\theta_{C2} \\theta_{C3}")

gammaC_1 = symbols("\\gamma_{C1}")  # revolute n chain m
gammaC_2 = symbols("\\gamma_{C2}")
gammaC_3 = symbols("\\gamma_{C3}")

Rz120 = Matrix(
    [
        [cos(2*pi/3), -sin(2*pi/3), 0],
        [sin(2*pi/3), cos(2*pi/3), 0],
        [0, 0, 1],
    ]
)

# Position of chain base frame expressed in inertial frame
A2 = Matrix([-D, 0, 0])
B2 = A2 + l_21 * Matrix([cos(thetaA_2), 0, sin(thetaA_2)])
C2 = B2 + l_22 * Matrix([cos(thetaB_2), 0, sin(thetaB_2)])
E2_chain = C2 + l_23 * Matrix([cos(thetaC_2) * cos(gammaC_2), cos(thetaC_2) * sin(gammaC_2), sin(thetaC_2)])

A1 = Rz120*A2
B1 = A1 + Rz120*(l_11 * Matrix([cos(thetaA_1), 0, sin(thetaA_1)]))
C1 = B1 + Rz120*(l_12 * Matrix([cos(thetaB_1), 0, sin(thetaB_1)]))
E1_chain = C1 + Rz120*(l_13 * Matrix([cos(thetaC_1) * cos(gammaC_1), cos(thetaC_1) * sin(gammaC_1), sin(thetaC_1)]))

A3 = Rz120.T*A2
B3 = A3 + Rz120.T*(l_31 * Matrix([cos(thetaA_3), 0, sin(thetaA_3)]))
C3 = B3 + Rz120.T*(l_32 * Matrix([cos(thetaB_3), 0, sin(thetaB_3)]))
E3_chain = C3 + Rz120.T*(l_33 * Matrix([cos(thetaC_3) * cos(gammaC_3), cos(thetaC_3) * sin(gammaC_3), sin(thetaC_3)]))

z = symbols("z")  # distance from o to A
alpha, beta, gamma = symbols("alpha beta gamma")

A = Matrix([0, 0, z])

# Rotation matrices for 3-1-2 rotation (Z,X,Y)
R_alpha = Matrix([[1, 0, 0],
                  [0, cos(alpha), -sin(alpha)],
                  [0, sin(alpha), cos(alpha)]])

R_beta = Matrix([[cos(beta), 0, sin(beta)],
                 [0, 1, 0],
                 [-sin(beta), 0, cos(beta)]])

R_gamma = Matrix([[cos(gamma), -sin(gamma), 0],
                  [sin(gamma), cos(gamma), 0],
                  [0, 0, 1]])

# Rotation to express vector in base frame to vector in rotated frame
R = R_gamma * R_beta * R_alpha
E_o = A + R * Matrix([0, 0, H])

# Define the points of the prismoid
E1_ee = E_o + R * Matrix([B / 2, -B * np.sqrt(3) / 2, 0])
E2_ee = E_o + R * Matrix([-B, 0, 0])
E3_ee = E_o + R * Matrix([B / 2, B * np.sqrt(3) / 2, 0])

p1 = 0.1  # fixed distance between C1 and A
p2 = p1  # fixed distance between C2 and A
p3 = p1  # fixed distance between C3 and A

f = Matrix.vstack(
    E1_ee - E1_chain,
    E2_ee - E2_chain,
    E3_ee - E3_chain,
    Matrix([(C1 - A).dot(C1 - A) - p1**2]),
    Matrix([(C2 - A).dot(C2 - A) - p2**2]),
    Matrix([(C3 - A).dot(C3 - A) - p3**2])
)

fk_unknowns = [thetaB_2, thetaB_3, thetaC_1, thetaC_2, thetaC_3, gammaC_1, gammaC_2, gammaC_3, alpha, beta, gamma, z]
ik_unknowns = [thetaA_1, thetaA_2, thetaA_3, thetaB_1, thetaB_2, thetaB_3, thetaC_1, thetaC_2, thetaC_3, gammaC_1, gammaC_2, gammaC_3]

J_fk = f.jacobian(fk_unknowns)
J_ik = f.jacobian(ik_unknowns)

# now, solve kinematics
geometric_params = {
    l_11: 0.100,  # m
    l_12: 0.100,  # m
    l_13: 0.050,  # m
    l_21: 0.100,  # m
    l_22: 0.100,  # m
    l_23: 0.050,  # m
    l_31: 0.100,  # m
    l_32: 0.100,  # m
    l_33: 0.050,  # m
    B:    0.060,  # m
    D:    0.120,
    H:    0.060,
}

# Apply only geometric parameters to keep joint variables symbolic
E1_chain_geo = E1_chain.subs(geometric_params)
E2_chain_geo = E2_chain.subs(geometric_params)
E3_chain_geo = E3_chain.subs(geometric_params)
C1_geo = C1.subs(geometric_params)
C2_geo = C2.subs(geometric_params)
C3_geo = C3.subs(geometric_params)
B1_geo = B1.subs(geometric_params)
B2_geo = B2.subs(geometric_params)
B3_geo = B3.subs(geometric_params)
E1_ee_geo = E1_ee.subs(geometric_params)
E2_ee_geo = E2_ee.subs(geometric_params)
E3_ee_geo = E3_ee.subs(geometric_params)
A_geo = A.subs(geometric_params)
A1_geo = A1.subs(geometric_params)
A2_geo = A2.subs(geometric_params)
A3_geo = A3.subs(geometric_params)
f_geo = f.subs(geometric_params)
J_fk_geo = J_fk.subs(geometric_params)
J_ik_geo = J_ik.subs(geometric_params)


# --- Lambdified callables (geometric params baked in, flat np.ndarray output) ---

_to3 = lambda f_: lambda *a: np.asarray(f_(*a), dtype=np.float64).ravel()
_to12 = lambda f_: lambda *a: np.asarray(f_(*a), dtype=np.float64).ravel()
_to12x12 = lambda f_: lambda *a: np.asarray(f_(*a), dtype=np.float64).reshape((12, 12))

A1_val = np.asarray(A1_geo.tolist(), dtype=np.float64).ravel()
A2_val = np.asarray(A2_geo.tolist(), dtype=np.float64).ravel()
A3_val = np.asarray(A3_geo.tolist(), dtype=np.float64).ravel()

fn_A        = _to3(lambdify([z], A_geo, "numpy", cse=True))
fn_B1       = _to3(lambdify([thetaA_1], B1_geo, "numpy", cse=True))
fn_B2       = _to3(lambdify([thetaA_2], B2_geo, "numpy", cse=True))
fn_B3       = _to3(lambdify([thetaA_3], B3_geo, "numpy", cse=True))
fn_C1       = _to3(lambdify([thetaA_1, thetaB_1], C1_geo, "numpy", cse=True))
fn_C2       = _to3(lambdify([thetaA_2, thetaB_2], C2_geo, "numpy", cse=True))
fn_C3       = _to3(lambdify([thetaA_3, thetaB_3], C3_geo, "numpy", cse=True))
fn_E1_chain = _to3(lambdify([thetaA_1, thetaB_1, thetaC_1, gammaC_1], E1_chain_geo, "numpy", cse=True))
fn_E2_chain = _to3(lambdify([thetaA_2, thetaB_2, thetaC_2, gammaC_2], E2_chain_geo, "numpy", cse=True))
fn_E3_chain = _to3(lambdify([thetaA_3, thetaB_3, thetaC_3, gammaC_3], E3_chain_geo, "numpy", cse=True))
fn_E1_ee    = _to3(lambdify([alpha, beta, gamma, z], E1_ee_geo, "numpy", cse=True))
fn_E2_ee    = _to3(lambdify([alpha, beta, gamma, z], E2_ee_geo, "numpy", cse=True))
fn_E3_ee    = _to3(lambdify([alpha, beta, gamma, z], E3_ee_geo, "numpy", cse=True))
fn_f        = _to12(lambdify(
    [thetaA_1, thetaA_2, thetaA_3,
     thetaB_1, thetaB_2, thetaB_3,
     thetaC_1, thetaC_2, thetaC_3,
     gammaC_1, gammaC_2, gammaC_3,
     alpha, beta, gamma, z],
    f_geo, "numpy", cse=True))
fn_J_fk = _to12x12(lambdify(
    [thetaA_1, thetaA_2, thetaA_3, thetaB_1] + fk_unknowns,
    J_fk_geo, "numpy", cse=True))
fn_J_ik = _to12x12(lambdify(
    ik_unknowns + [alpha, beta, gamma, z],
    J_ik_geo, "numpy", cse=True))


In [ ]:
import numpy as np
import plotly.graph_objects as go

def visualize_rrur(q, show_figure=False, save_html=False, showlegend=False,
                   width=800, height=600,
                   x_range=(-.300, .200), y_range=(-.250, .250), z_range=(-.010, .510),
                   title='RRUR Mechanism'):
    """q: (16,) array ordered [θA1 θA2 θA3 θB1 θB2 θB3 θC1 θC2 θC3 γC1 γC2 γC3 α β γ z]"""
    tA1, tA2, tA3, tB1, tB2, tB3, tC1, tC2, tC3, gC1, gC2, gC3, al, be, ga, zv = q

    # Chain joint positions
    b1 = fn_B1(tA1);  c1 = fn_C1(tA1, tB1);  e1c = fn_E1_chain(tA1, tB1, tC1, gC1)
    b2 = fn_B2(tA2);  c2 = fn_C2(tA2, tB2);  e2c = fn_E2_chain(tA2, tB2, tC2,gC2)
    b3 = fn_B3(tA3);  c3 = fn_C3(tA3, tB3);  e3c = fn_E3_chain(tA3, tB3, tC3, gC3)

    # End-effector points
    a  = fn_A(zv)
    e1 = fn_E1_ee(al, be, ga, zv)
    e2 = fn_E2_ee(al, be, ga, zv)
    e3 = fn_E3_ee(al, be, ga, zv)

    def _chain_trace(pts, color, name):
        pts = np.array(pts)
        return go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2],
                            mode='lines+markers', line=dict(color=color, width=4),
                            marker=dict(size=3), name=name)

    def _line_trace(p0, p1, **kw):
        return go.Scatter3d(x=[p0[0],p1[0]], y=[p0[1],p1[1]], z=[p0[2],p1[2]],
                            mode='lines', showlegend=False, **kw)

    fig = go.Figure(data=[
        # Chains: A → B → C → E
        _chain_trace([A1_val, b1, c1, e1c], 'red',   'Chain 1'),
        _chain_trace([A2_val, b2, c2, e2c], 'green', 'Chain 2'),
        _chain_trace([A3_val, b3, c3, e3c], 'blue',  'Chain 3'),
        # EE triangle
        _line_trace(e1, e2, line=dict(color='black', width=3)),
        _line_trace(e2, e3, line=dict(color='black', width=3)),
        _line_trace(e3, e1, line=dict(color='black', width=3)),
        # A → Ei struts
        _line_trace(a, e1, line=dict(color='black', dash='dash')),
        _line_trace(a, e2, line=dict(color='black', dash='dash')),
        _line_trace(a, e3, line=dict(color='black', dash='dash')),
        # A marker
        go.Scatter3d(x=[a[0]], y=[a[1]], z=[a[2]],
                     mode='markers', marker=dict(color='black', size=5, symbol='x'), name='A'),
    ])

    fig.update_layout(
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
                   xaxis=dict(range=list(x_range)),
                   yaxis=dict(range=list(y_range)),
                   zaxis=dict(range=list(z_range)),
                   aspectmode='manual', aspectratio=dict(x=1, y=1, z=1)),
        title=title, width=width, height=height, showlegend=showlegend)

    if show_figure:
        fig.show()
    if save_html:
        fig.write_html(save_html)
    return fig

In [ ]:
def fk(thetaA_1, thetaA_2, thetaA_3, thetaB_1, x0):
  def evaluate_fk(q_fk):
    return fn_f(thetaA_1, thetaA_2, thetaA_3, thetaB_1, *q_fk)
  return root(evaluate_fk, x0)




initial_guess = np.array([np.pi/3, np.pi/3,      # θB2, θB3
               np.pi/4, np.pi/4, np.pi/4,        # θC1, θC2, θC3
               0.0, 0.0, 0.0,                    # γC1, γC2, γC3
               0.0, 0.0, 0.0, 0.1])              # α, β, γ, z
fk_angles = np.array([2*np.pi/3, 2*np.pi/3, 2*np.pi/3, np.pi/6])




q_fk = fk(*fk_angles, initial_guess)['x']
fig = visualize_rrur(np.array([2*np.pi/3, 2*np.pi/3, 2*np.pi/3, np.pi/6, *q_fk]), show_figure=True)



In [ ]:
def ik(alpha, beta, gamma, z, x0):
  def evaluate_ik(q_ik):
    return fn_f(*q_ik, alpha, beta, gamma, z)
  return root(evaluate_ik, x0, tol=1e-4)


initial_guess = np.array([5*np.pi/6, 5*np.pi/6, 5*np.pi/6,        # θA1, θA2, θA3
                np.pi/3, np.pi/3, np.pi/3,                        # θB1, θB2, θB3
                np.pi/6, np.pi/6, np.pi/6,                        # θC1, θC2, θC3
                0.0, 0.0, 0.0,                                    # γC1, γC2, γC3
                ])

ik_positions = np.array([0, 0, np.pi/6, .07])

q_ik = ik(*ik_positions, initial_guess)['x']
print(ik(*ik_positions, initial_guess)['success'])
fig = visualize_rrur(np.array([*q_ik, *ik_positions]), show_figure=True)
q_ik

True


array([ 2.24079398,  2.24079398,  2.24079398,  0.37798726,  0.37798726,
        0.37798726,  0.29867331,  0.29867331,  0.29867331, -0.67871551,
       -0.67871551, -0.67871551])

In [ ]:
import time
from scipy.stats import qmc
from tqdm.notebook import tqdm

alpha_range = np.deg2rad(60)
beta_range = np.deg2rad(60)
gamma_range = np.deg2rad(60)
z_range = 0.1

home_pose = np.array([0, 0, np.pi/6, .07])
# home_pose = np.array([0, 0, 0, .07])

num_samples = 16384 ## will round to the nearest power of 2

m_power = int(np.round(np.log2(num_samples))) # Sobol requires power of 2 for uniform stratification
sampler = qmc.Sobol(d=4, scramble=True)
poses = home_pose + qmc.scale(sampler.random_base2(m=m_power), -np.array([alpha_range, beta_range, gamma_range, z_range]), np.array([alpha_range, beta_range, gamma_range, z_range]))
sorted_poses = poses[np.lexsort(poses.T[::-1])] # Sorts z, then gamma, then beta, then alpha


workspace = []
with tqdm(total=num_samples, desc="Validating Workspace") as pbar:
  pbar_count = 0
  prev_pose = sorted_poses[0]
  warmstart = initial_guess
  for pose in sorted_poses:
    if np.linalg.norm(pose - prev_pose) > 0.5: # big jump in sorted array, reset warmstart
      warmstart = initial_guess

    sol = ik(*pose, warmstart)

    if sol['success']:
      workspace.append(pose)
      warmstart = sol['x']

    prev_pose = pose
    pbar_count += 1
    if pbar_count % 100 == 0: pbar.update(100)

e0_collection = []
for element in workspace:
  e1 = fn_E1_ee(*element)
  e2 = fn_E2_ee(*element)
  e3 = fn_E3_ee(*element)
  e0 = (e1 + e2 + e3)/3
  e0_collection.append(e0)

e0_matrix = np.array(e0_collection)
X_coords = e0_matrix[:, 0]
Y_coords = e0_matrix[:, 1]
Z_coords = e0_matrix[:, 2]

print(len(workspace))

Validating Workspace:   0%|          | 0/16384 [00:00<?, ?it/s]

2144


In [ ]:
fig = visualize_rrur(np.array([2*np.pi/3, 2*np.pi/3, 2*np.pi/3, np.pi/6, *q_fk]))

fig.add_trace(go.Scatter3d(
    x=X_coords,
    y=Y_coords,
    z=Z_coords,
    mode='markers',
    marker=dict(
        size=2,                 # Small markers for dense clouds
        color=Z_coords,         # Color mapped to Z-height
        colorscale='Viridis',
        opacity=0.6             # Transparency for internal density
    )
))

fig.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

# 1. Initialize a history dictionary if it doesn't exist yet
# (This lets you change parameters in your previous cell, re-run it, and compare results here)
if 'workspace_history' not in locals():
    workspace_history = {}

# --- CRITICAL: CHANGE THIS STRING EACH TIME YOU CHANGE YOUR LINK LENGTHS ---
current_design_name = "Baseline_L1_0.3_L2_0.3"
# --------------------------------------------------------------------------

# 2. Extract metrics from your existing variables
successful_samples = len(workspace)
ik_success_rate = (successful_samples / num_samples) * 100

# Calculate the physical volume enclosed by your end-effector points
try:
    hull = ConvexHull(e0_matrix)
    workspace_volume = hull.volume
except Exception as e:
    print(f"Could not compute Convex Hull (likely too few points or flat plane): {e}")
    workspace_volume = 0.0

# Calculate raw bounding box ranges for reference
x_range = np.ptp(X_coords) if len(X_coords) > 0 else 0
y_range = np.ptp(Y_coords) if len(Y_coords) > 0 else 0
z_range_actual = np.ptp(Z_coords) if len(Z_coords) > 0 else 0

# 3. Store the results for comparison
workspace_history[current_design_name] = {
    "Volume (m³)": workspace_volume,
    "IK Success Rate (%)": ik_success_rate,
    "X Range (m)": x_range,
    "Y Range (m)": y_range,
    "Z Range (m)": z_range_actual,
    "Points": e0_matrix.copy()
}

# 4. Print Professional Analysis Report
print("="*50)
print(f" ROBOTICS WORKSPACE METRICS FOR: {current_design_name}")
print("="*50)
print(f"Reachable Volume (Convex Hull) : {workspace_volume:.6f} m³")
print(f"Global Workspace Density (GCI) : {ik_success_rate:.2f}% (Valid joint states / total samples)")
print(f"Bounding Box Dimensions         : X: {x_range:.3f}m, Y: {y_range:.3f}m, Z: {z_range_actual:.3f}m")
print("-"*50)

# 5. Print Comparison Table if you have multiple runs stored
if len(workspace_history) > 1:
    print("\n" + "="*50)
    print(" DESIGN PARAMETER COMPARISON")
    print("="*50)
    print(f"{'Design Name':<30} | {'Volume (m³)':<12} | {'IK Success %':<12}")
    print("-"*55)
    for name, metrics in workspace_history.items():
        print(f"{name:<30} | {metrics['Volume (m³)']:<12.6f} | {metrics['IK Success Rate (%)']:<12.2f}")
    print("="*55)

    # Plot visual comparison of the workspaces
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    colors = plt.cm.rainbow(np.linspace(0, 1, len(workspace_history)))

    for (name, metrics), color in zip(workspace_history.items(), colors):
        pts = metrics["Points"]
        if len(pts) > 0:
            # Subsample points if it lags your notebook visualization
            step = max(1, len(pts) // 1000)
            ax.scatter(pts[::step, 0], pts[::step, 1], pts[::step, 2], alpha=0.4, s=2, label=name, color=color)

    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')
    ax.set_zlabel('Z Position (m)')
    ax.set_title('Workspace Parameter Overlay Comparison')
    ax.legend()
    plt.show()

 ROBOTICS WORKSPACE METRICS FOR: Baseline_L1_0.3_L2_0.3
Reachable Volume (Convex Hull) : 0.001243 m³
Global Workspace Density (GCI) : 4.24% (Valid joint states / total samples)
Bounding Box Dimensions         : X: 0.119m, Y: 0.112m, Z: 0.233m
--------------------------------------------------


In [ ]:
initial_guess = np.array([5*np.pi/6, 5*np.pi/6, 5*np.pi/6,        # θA1, θA2, θA3
                np.pi/3, np.pi/3+0.01, np.pi/3,                        # θB1, θB2, θB3
                np.pi/6, np.pi/6, np.pi/6,                        # θC1, θC2, θC3
                0.0, 0.0, 0.0,                                    # γC1, γC2, γC3
                ])

pose = np.array([0,np.deg2rad(0),np.deg2rad(45),.05])
q_ik = ik(*pose, initial_guess)['x']
print(ik(*pose, initial_guess)['success'])
fig = visualize_rrur(np.array([*q_ik, *pose]), show_figure=True)

np.linalg.norm(fn_f(*q_ik, *pose))

True


np.float64(1.8738531573295434e-05)

In [ ]:
pose = np.array([0,np.deg2rad(45),np.deg2rad(90),.05])

start_time = time.perf_counter()
for i in range(0,1000):
  ik(*pose, initial_guess)
print(time.perf_counter() - start_time)

10.874518051999985


In [ ]:
pose = np.array([0,np.deg2rad(0),np.deg2rad(45),.05])

start_time = time.perf_counter()
for i in range(0,1000):
  ik(*pose, initial_guess)
print(time.perf_counter() - start_time)

8.470050510999954
